In [1]:
import pandas as pd
import numpy as np

In [2]:
import micropip
await micropip.install('openpyxl')

In [3]:
order_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]
df_orders = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in order_files],
    ignore_index=True
)

sheets = pd.read_excel("data/products.xlsx", sheet_name=["products", "categories"])
df_products = sheets["products"]
df_categories = sheets["categories"]

df_events_records = pd.read_json("data/events_records.json", orient="records")
df_events_lines = pd.read_json("data/events_lines.json", lines=True)
df_events = pd.concat([df_events_records, df_events_lines], ignore_index=True)
df_events = df_events.drop_duplicates(subset=["event_id"]).reset_index(drop=True)
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

print("Заказы:", df_orders.shape)
print("Товары:", df_products.shape)
print("Категории:", df_categories.shape)
print("События:", df_events.shape)

Заказы: (15000, 7)
Товары: (30, 5)
Категории: (5, 3)
События: (2001, 2005)


In [4]:
orders_paid = df_orders[df_orders['status'] == 'paid'].copy()

user_metrics = (orders_paid.groupby('user_id')
                .agg(
                    orders_count=('order_id', 'nunique'),
                    orders_total_amount=('price', lambda s: (s * orders_paid.loc[s.index, 'quantity']).sum()),
                    avg_order_price=('price', 'mean'),
                    first_order=('order_datetime', 'min'),
                    last_order=('order_datetime', 'max'),
                    distinct_products=('product_id', 'nunique')
                )
                .reset_index())

print("Метрики по заказам:")
print(user_metrics.head())
print("\nВсего юзеров с заказами:", len(user_metrics))

Метрики по заказам:
   user_id  orders_count  orders_total_amount  avg_order_price  \
0     1001           142           3539635.84      9635.950493   
1     1002           101           2196505.04     10002.740693   
2     1003           133           3400406.47     10186.671429   
3     1004           125           3313341.75     10076.816000   
4     1005           115           2484426.17      9232.982696   

          first_order          last_order  distinct_products  
0 2024-01-02 02:51:37 2024-03-31 21:32:19                 30  
1 2024-01-02 12:57:34 2024-03-31 19:45:16                 29  
2 2024-01-02 02:02:44 2024-03-31 19:36:26                 30  
3 2024-01-01 00:19:34 2024-03-31 20:01:40                 30  
4 2024-01-01 01:11:51 2024-03-31 18:16:10                 30  

Всего юзеров с заказами: 100


In [5]:
events_by_user_extra = (df_events.groupby("user_id")
                        .agg(
                            events_total=("event_id", "count"),
                            events_page_view=("event_type", lambda s: (s == "page_view").sum()),
                            events_click=("event_type", lambda s: (s == "click").sum()),
                            events_add_to_cart=("event_type", lambda s: (s == "add_to_cart").sum()),
                            events_purchase=("event_type", lambda s: (s == "purchase").sum()),
                            first_event=("event_time", "min"),
                            last_event=("event_time", "max"),
                        ))
print("Агрегаты событий:")
print(events_by_user_extra.head())

Агрегаты событий:
         events_total  events_page_view  events_click  events_add_to_cart  \
user_id                                                                     
1001.0             29                 3             9                   4   
1002.0             25                 4             6                   3   
1003.0             21                 3             3                   2   
1004.0             22                 2             3                   4   
1005.0             21                 2             5                   4   

         events_purchase         first_event          last_event  
user_id                                                           
1001.0                11 2024-02-01 20:18:15 2024-03-31 22:29:02  
1002.0                 8 2024-02-05 07:27:49 2024-03-31 21:24:08  
1003.0                 8 2024-02-09 07:00:19 2024-03-29 20:53:24  
1004.0                 9 2024-02-03 21:50:28 2024-03-27 04:58:11  
1005.0                 7 2024-02-02 13:5

In [6]:
user_features = user_metrics.merge(events_by_user_extra,
                                   on="user_id", how="outer")

user_features["first_activity"] = user_features[["first_order", "first_event"]].min(axis=1)
user_features["last_activity"]  = user_features[["last_order",  "last_event"]].max(axis=1)
user_features["lifetime_days"]  = (user_features["last_activity"]
                                   - user_features["first_activity"]).dt.days

print("Интегрированные фичи:")
print(user_features.head())
print("\nВсего ющеров:", len(user_features))

Интегрированные фичи:
   user_id  orders_count  orders_total_amount  avg_order_price  \
0     1001           142           3539635.84      9635.950493   
1     1002           101           2196505.04     10002.740693   
2     1003           133           3400406.47     10186.671429   
3     1004           125           3313341.75     10076.816000   
4     1005           115           2484426.17      9232.982696   

          first_order          last_order  distinct_products  events_total  \
0 2024-01-02 02:51:37 2024-03-31 21:32:19                 30            29   
1 2024-01-02 12:57:34 2024-03-31 19:45:16                 29            25   
2 2024-01-02 02:02:44 2024-03-31 19:36:26                 30            21   
3 2024-01-01 00:19:34 2024-03-31 20:01:40                 30            22   
4 2024-01-01 01:11:51 2024-03-31 18:16:10                 30            21   

   events_page_view  events_click  events_add_to_cart  events_purchase  \
0                 3             9     

In [7]:
count_cols = ['orders_count', 'orders_total_amount', 'distinct_products',
              'events_total', 'events_page_view', 'events_click',
              'events_add_to_cart', 'events_purchase']
for col in count_cols:
    user_features[col] = user_features[col].fillna(0)

user_features['avg_order_price'] = user_features['avg_order_price'].fillna(0)

user_features['lifetime_days'] = user_features['lifetime_days'].fillna(0).astype(int)

user_features['purchase_event_ratio'] = np.where(
    user_features['events_total'] > 0,
    user_features['events_purchase'] / user_features['events_total'],
    0.0
)

user_features['orders_per_day'] = np.where(
    user_features['lifetime_days'] > 0,
    user_features['orders_count'] / user_features['lifetime_days'],
    user_features['orders_count']
)

user_features['days_to_first_order'] = (
    (user_features['first_order'] - user_features['first_event']).dt.days
).fillna(-1).astype(int) 

print("После заполнения пропусков и производных фич:")
print(user_features.head())

После заполнения пропусков и производных фич:
   user_id  orders_count  orders_total_amount  avg_order_price  \
0     1001           142           3539635.84      9635.950493   
1     1002           101           2196505.04     10002.740693   
2     1003           133           3400406.47     10186.671429   
3     1004           125           3313341.75     10076.816000   
4     1005           115           2484426.17      9232.982696   

          first_order          last_order  distinct_products  events_total  \
0 2024-01-02 02:51:37 2024-03-31 21:32:19                 30            29   
1 2024-01-02 12:57:34 2024-03-31 19:45:16                 29            25   
2 2024-01-02 02:02:44 2024-03-31 19:36:26                 30            21   
3 2024-01-01 00:19:34 2024-03-31 20:01:40                 30            22   
4 2024-01-01 01:11:51 2024-03-31 18:16:10                 30            21   

   events_page_view  events_click  events_add_to_cart  events_purchase  \
0             

In [8]:
feature_cols = [
    # --- события ---
    'events_total',
    'events_page_view',
    'events_click',
    'events_add_to_cart',
    'events_purchase',
    'purchase_event_ratio',
    'lifetime_days',
    'days_to_first_order',
    'orders_per_day',
]

df_model = user_features[user_features['orders_count'] > 0].copy()

print("Фичи:")
print(df_model[['user_id'] + feature_cols].head())
print("\nРазмер датасета:", df_model.shape)

Фичи:
   user_id  events_total  events_page_view  events_click  events_add_to_cart  \
0     1001            29                 3             9                   4   
1     1002            25                 4             6                   3   
2     1003            21                 3             3                   2   
3     1004            22                 2             3                   4   
4     1005            21                 2             5                   4   

   events_purchase  purchase_event_ratio  lifetime_days  days_to_first_order  \
0               11              0.379310             89                  -31   
1                8              0.320000             89                  -34   
2                8              0.380952             89                  -39   
3                9              0.409091             90                  -34   
4                7              0.333333             90                  -33   

   orders_per_day  
0        1.5

In [9]:
df_model['is_repeated_buyer'] = (df_model['orders_count'] > 1).astype(int)

print("Распределение таргета:")
print(df_model['is_repeated_buyer'].value_counts().rename_axis('is_repeated_buyer').reset_index(name='count'))
print(f"\nДоля повторных покупателей: {df_model['is_repeated_buyer'].mean():.2%}")

Распределение таргета:
   is_repeated_buyer  count
0                  1    100

Доля повторных покупателей: 100.00%


In [10]:
from sklearn.model_selection import train_test_split

X = df_model[feature_cols].copy()
y = df_model['is_repeated_buyer'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| test:", X_test.shape)
print("\nБаланс классов в train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nБаланс классов в test:")
print(y_test.value_counts(normalize=True).round(3))

Train: (75, 9) | test: (25, 9)

Баланс классов в train:
is_repeated_buyer
1    1.0
Name: proportion, dtype: float64

Баланс классов в test:
is_repeated_buyer
1    1.0
Name: proportion, dtype: float64


In [11]:
train_out = pd.concat([X_train.reset_index(drop=True),
                       y_train.reset_index(drop=True)], axis=1)
test_out  = pd.concat([X_test.reset_index(drop=True),
                       y_test.reset_index(drop=True)], axis=1)

train_out.to_csv('data/train_features.csv', index=False)
test_out.to_csv('data/test_features.csv', index=False)

print("Сохранено:")
print(" - data/train_features.csv:", train_out.shape)
print(" - data/test_features.csv: ", test_out.shape)
print("\nКолонки:", list(train_out.columns))

Сохранено:
 - data/train_features.csv: (75, 10)
 - data/test_features.csv:  (25, 10)

Колонки: ['events_total', 'events_page_view', 'events_click', 'events_add_to_cart', 'events_purchase', 'purchase_event_ratio', 'lifetime_days', 'days_to_first_order', 'orders_per_day', 'is_repeated_buyer']
